# Post-deforestation


In [ ]:
# ============================================================
# 0. Imports
# ============================================================
import numpy as np
import pandas as pd
import xarray as xr
import json
from pathlib import Path
import matplotlib.pyplot as plt

from hbv_bmi import HBV_Bmi


# ============================================================
# 1. Paths and periods
# ============================================================
data_dir = Path("./Data")

precip_nc = data_dir / "manning_ERA5_precip_daily.nc"
evap_nc   = data_dir / "manning_ERA5_evap_daily.nc"
discharge_file = data_dir / "5202080_Q_Day.Cmd.txt"

shape_area = 6642 * 1e6  # m²

pre_start = "2014-01-01"
pre_end   = "2019-10-01"

exp_start = pre_start
exp_end   = pre_end


# ============================================================
# 2. Load discharge and clean
# ============================================================
obs = pd.read_csv(
    discharge_file,
    delimiter=';',
    skiprows=36,
    header=0,
    encoding='cp1252'
)
obs.columns = ["Date", "Time", "Q"]
obs["Q"] = pd.to_numeric(obs["Q"], errors="coerce")
obs = obs.dropna(subset=["Q"])
obs["Date"] = pd.to_datetime(obs["Date"])
obs = obs.set_index("Date").drop(columns=["Time"]).sort_index()

# Convert m³/s → mm/day if needed
if obs["Q"].max() > 50:
    obs["Q"] = obs["Q"] * 86400 / shape_area * 1000.0

obs.loc[obs["Q"] > 4000, "Q"] = np.nan
obs.loc[obs["Q"] < 0, "Q"] = np.nan
obs = obs.loc[pre_start:pre_end]


# ============================================================
# 3. Precompute time index from forcing
# ============================================================
ds_precip = xr.open_dataset(precip_nc)
time_index = pd.to_datetime(ds_precip["time"].values)
ds_precip.close()

mask = (time_index >= pd.to_datetime(pre_start)) & (time_index <= pd.to_datetime(pre_end))
time_index = time_index[mask]

obs = obs.reindex(time_index)


# ============================================================
# 4. Objective function
# ============================================================
def compute_metrics(sim, obs):
    sim = sim.copy()
    obs = obs.copy()

    # Fill isolated NaNs in obs
    for i in range(1, len(obs) - 1):
        if np.isnan(obs[i]) and not np.isnan(obs[i - 1]) and not np.isnan(obs[i + 1]):
            obs[i] = 0.5 * (obs[i - 1] + obs[i + 1])

    mask = ~np.isnan(obs)
    sim = sim[mask]
    obs = obs[mask]

    if len(obs) < 3:
        return np.nan, np.nan, np.nan

    sim_c = np.clip(sim, 1e-6, None)
    obs_c = np.clip(obs, 1e-6, None)

    nse = 1 - np.sum((sim - obs)**2) / np.sum((obs - obs.mean())**2)
    lognse = 1 - np.sum((np.log(sim_c) - np.log(obs_c))**2) / \
                  np.sum((np.log(obs_c) - np.log(obs_c).mean())**2)
    De = np.sqrt((1 - nse)**2 + (1 - lognse)**2)

    return nse, lognse, De


# ============================================================
# 5. Ensemble setup
# ============================================================
p_min = np.array([0., 0.4,  40., 1.0, 0.001, 1.,  0.01, 0.0001])
p_max = np.array([8., 0.8, 800., 2.5, 0.3,   3.,  0.1,  0.01])

N = 40  # fast and stable
parameters = np.zeros((8, N))

np.random.seed(6)
for i in range(8):
    parameters[i, :] = np.random.uniform(p_min[i], p_max[i], N)


# ============================================================
# 6. One reusable JSON config file
# ============================================================
config_path = Path("./hbv_config.json")

base_config = {
    "precipitation_file": str(precip_nc),
    "potential_evaporation_file": str(evap_nc),
    "initial_storage": "0,100,0,5",
    "parameters": ""
}

with open(config_path, "w") as f:
    json.dump(base_config, f)


# ============================================================
# 7. Run ensemble (calibration only)
# ============================================================
results = []

for n in range(N):
    par = parameters[:, n]

    base_config["parameters"] = ",".join(str(p) for p in par)
    with open(config_path, "w") as f:
        json.dump(base_config, f)

    model = HBV_Bmi()
    model.initialize(str(config_path))

    n_steps = model.end_timestep
    Q_sim = np.zeros(n_steps)

    for t in range(n_steps):
        model.update()
        Q_sim[t] = model.Q

    model_time = pd.date_range(pre_start, periods=n_steps, freq="D")
    sim_df = pd.DataFrame({"model": Q_sim}, index=model_time)
    sim_df = sim_df.loc[pre_start:pre_end]

    sim_arr = sim_df["model"].values
    obs_arr = obs["Q"].values

    nse, lognse, De = compute_metrics(sim_arr, obs_arr)
    results.append((nse, lognse, De))


results = np.array(results)


# ============================================================
# 8. Extract best parameter sets
# ============================================================
best_nse_idx = np.nanargmax(results[:, 0])
best_lognse_idx = np.nanargmax(results[:, 1])
best_De_idx = np.nanargmin(results[:, 2])

best_par_nse = parameters[:, best_nse_idx]
best_par_lognse = parameters[:, best_lognse_idx]
best_par_De = parameters[:, best_De_idx]

print("\nBest NSE parameters:\n", best_par_nse)
print("Best logNSE parameters:\n", best_par_lognse)
print("Best De parameters:\n", best_par_De)

print("\nBest NSE =", results[best_nse_idx, 0])
print("Best logNSE =", results[best_lognse_idx, 1])
print("Best De =", results[best_De_idx, 2])
